In [1]:
from google.colab import drive
drive.mount('/content/drive')

# Copy model files from Drive to local
import shutil
src = '/content/drive/MyDrive/Colab Notebooks/Gsoc 2026/ML4sci/'
shutil.copy(src + 'best_lstm.pth', 'best_lstm.pth')
shutil.copy(src + 'best_transformer.pth', 'best_transformer.pth')
shutil.copy(src + 'notebook1_results.pkl', 'notebook1_results.pkl')
print("✅ Models loaded from Drive!")

Mounted at /content/drive
✅ Models loaded from Drive!


# FASEROH: Beam Search & Experimental Findings
## Notebook 2 — Improving Inference and Understanding Failure Modes

**Author:** Vishal Lohiya | IIT Jodhpur | GSoC 2026 — ML4Sci

---

### What This Notebook Does

In Notebook 1, we discovered that the Transformer has **low validation loss but terrible test accuracy** — the exposure bias problem. This notebook explores two solutions:

1. **Beam Search Decoding** — instead of greedily picking one token at a time, explore multiple candidate sequences and pick the best one. This fixes the problem at inference time without retraining.

2. **Teacher Forcing Decay for Transformer** — the same technique that made the LSTM robust. We show that this approach **fails catastrophically** for Transformers and explain why.

### Why Beam Search Over Greedy?

**Greedy decoding** picks the single most probable token at each step. If one token is wrong, the error cascades — every subsequent token is generated from a corrupted context.

**Beam search** (width k=5) keeps the **top 5 candidate sequences** alive at each step. Think of it like navigating a maze:
- Greedy = always turn right. If one turn is wrong, you're lost.
- Beam search = explore 5 paths simultaneously. Even if one path hits a dead end, the others might find the exit.

```
Greedy (1 path):              Beam Search (5 paths):
1 + x + x²/3  ✗              1 + x + x²/2  ✓  ← kept as candidate
     (wrong, stuck)           1 + x + x²/3  ✗  ← also explored
                              1 + x + x²/4  ✗  ← also explored
                              ...picks the best complete sequence
```

**Prerequisites:** Run Notebook 1 first to generate `best_lstm.pth`, `best_transformer.pth`, and the data files.

---
## 0. Setup

Load everything from Notebook 1.

In [2]:
# Install dependencies (uncomment if needed)
# !pip install torch sympy numpy pandas matplotlib scikit-learn

import re, time, random, pickle, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

import sympy as sp
from sympy import (Symbol, series, sin, cos, exp, log, tan, sqrt,
                   atan, asin, acos, sinh, cosh, tanh, Rational,
                   expand, zoo, nan, oo, I)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
x = Symbol('x')

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Device: {device}")

# Check that Notebook 1 outputs exist
required_files = ['best_lstm.pth', 'best_transformer.pth', 'notebook1_results.pkl']
missing = [f for f in required_files if not os.path.exists(f)]
if missing:
    print(f"\n⚠️  Missing files: {missing}")
    print("Please run Notebook 1 (01_LSTM_vs_Transformer.ipynb) first!")
else:
    print("\n✅ All Notebook 1 outputs found. Loading...")

Device: cpu

✅ All Notebook 1 outputs found. Loading...


In [3]:
# ============================================================
# Rebuild all classes and data (same as Notebook 1)
# We need these to load the saved model weights
# ============================================================

# --- Tokenizer (same as Notebook 1) ---
class MathTokenizer:
    PAD_TOKEN = '<PAD>'; SOS_TOKEN = '<SOS>'; EOS_TOKEN = '<EOS>'; UNK_TOKEN = '<UNK>'
    def __init__(self):
        self.token2idx = {}; self.idx2token = {}; self.vocab_size = 0
        for i, t in enumerate([self.PAD_TOKEN, self.SOS_TOKEN, self.EOS_TOKEN, self.UNK_TOKEN]):
            self.token2idx[t] = i; self.idx2token[i] = t
        self.vocab_size = 4
    def tokenize(self, s):
        s = s.replace(' ', '')
        raw = re.findall(r'(sin|cos|tan|exp|log|sqrt|sinh|cosh|tanh|asin|acos|atan|\*\*|\d+|\+|\-|\*|/|\(|\)|x)', s)
        tokens = []
        for t in raw:
            if t.isdigit() and len(t) > 1:
                for d in t: tokens.append(d)
            else: tokens.append(t)
        return tokens
    def build_vocab(self, exprs=None):
        for t in ['0','1','2','3','4','5','6','7','8','9','+','-','*','/','**','(',')','x',
                   'sin','cos','tan','exp','log','sqrt','sinh','cosh','tanh','asin','acos','atan']:
            if t not in self.token2idx:
                self.token2idx[t] = self.vocab_size; self.idx2token[self.vocab_size] = t; self.vocab_size += 1
        if exprs:
            for e in exprs:
                for t in self.tokenize(e):
                    if t not in self.token2idx:
                        self.token2idx[t] = self.vocab_size; self.idx2token[self.vocab_size] = t; self.vocab_size += 1
        return self
    def encode(self, s, add_sos=False, add_eos=False):
        ids = []
        if add_sos: ids.append(self.token2idx[self.SOS_TOKEN])
        for t in self.tokenize(s): ids.append(self.token2idx.get(t, self.token2idx[self.UNK_TOKEN]))
        if add_eos: ids.append(self.token2idx[self.EOS_TOKEN])
        return ids
    def decode(self, ids):
        tokens = [self.idx2token.get(i, self.UNK_TOKEN) for i in ids
                  if self.idx2token.get(i, '') not in [self.PAD_TOKEN, self.SOS_TOKEN, self.EOS_TOKEN]]
        r = ''; i = 0
        while i < len(tokens):
            t = tokens[i]
            if t.isdigit():
                n = t
                while i+1 < len(tokens) and tokens[i+1].isdigit(): i += 1; n += tokens[i]
                r += n
            elif t in ['+', '-']: r += ' ' + t + ' '
            else: r += t
            i += 1
        return r.strip()
    def pad_sequence(self, ids, ml):
        return ids[:ml] if len(ids) >= ml else ids + [self.token2idx[self.PAD_TOKEN]] * (ml - len(ids))

# --- Model classes (same as Notebook 1) ---
class LSTMEncoder(nn.Module):
    def __init__(self, vs, ed, hd, nl, do):
        super().__init__()
        self.embedding = nn.Embedding(vs, ed, padding_idx=0)
        self.lstm = nn.LSTM(ed, hd, nl, batch_first=True, dropout=do if nl>1 else 0, bidirectional=True)
        self.fc_hidden = nn.Linear(hd*2, hd); self.fc_cell = nn.Linear(hd*2, hd)
        self.dropout = nn.Dropout(do)
    def forward(self, src):
        out, (h, c) = self.lstm(self.dropout(self.embedding(src)))
        h = torch.tanh(self.fc_hidden(torch.cat([h[-2], h[-1]], dim=1))).unsqueeze(0)
        c = torch.tanh(self.fc_cell(torch.cat([c[-2], c[-1]], dim=1))).unsqueeze(0)
        return out, h, c

class BahdanauAttention(nn.Module):
    def __init__(self, hd):
        super().__init__()
        self.attn = nn.Linear(hd*3, hd); self.v = nn.Linear(hd, 1, bias=False)
    def forward(self, hidden, enc_out):
        h = hidden.squeeze(0).unsqueeze(1).repeat(1, enc_out.shape[1], 1)
        return torch.softmax(self.v(torch.tanh(self.attn(torch.cat([h, enc_out], dim=2)))).squeeze(2), dim=1)

class LSTMDecoder(nn.Module):
    def __init__(self, vs, ed, hd, nl, do):
        super().__init__()
        self.embedding = nn.Embedding(vs, ed, padding_idx=0)
        self.attention = BahdanauAttention(hd)
        self.lstm = nn.LSTM(ed+hd*2, hd, 1, batch_first=True)
        self.fc_out = nn.Linear(hd*3+ed, vs); self.dropout = nn.Dropout(do)
    def forward(self, tok, h, c, enc_out):
        emb = self.dropout(self.embedding(tok))
        aw = self.attention(h, enc_out).unsqueeze(1)
        ctx = torch.bmm(aw, enc_out)
        out, (h, c) = self.lstm(torch.cat([emb, ctx], dim=2), (h, c))
        return self.fc_out(torch.cat([out, ctx, emb], dim=2)).squeeze(1), h, c

class Seq2SeqLSTM(nn.Module):
    def __init__(self, vs, ed, hd, nl, do):
        super().__init__()
        self.encoder = LSTMEncoder(vs, ed, hd, nl, do)
        self.decoder = LSTMDecoder(vs, ed, hd, nl, do)
        self.vocab_size = vs
    def forward(self, src, tgt, tf=0.5):
        enc_out, h, c = self.encoder(src)
        B, T = tgt.shape[0], tgt.shape[1]
        outputs = torch.zeros(B, T, self.vocab_size).to(src.device)
        inp = tgt[:, 0:1]
        for t in range(1, T):
            pred, h, c = self.decoder(inp, h, c, enc_out)
            outputs[:, t] = pred
            inp = tgt[:, t:t+1] if random.random() < tf else pred.argmax(dim=1, keepdim=True)
        return outputs

class PositionalEncoding(nn.Module):
    def __init__(self, d, ml=500, do=0.1):
        super().__init__()
        self.dropout = nn.Dropout(do)
        pe = torch.zeros(ml, d)
        pos = torch.arange(0, ml, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d, 2).float() * (-np.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): return self.dropout(x + self.pe[:, :x.size(1)])

class Seq2SeqTransformer(nn.Module):
    def __init__(self, vs, dm, nh, nl, df, do):
        super().__init__()
        self.d_model = dm
        self.src_emb = nn.Embedding(vs, dm, padding_idx=0)
        self.tgt_emb = nn.Embedding(vs, dm, padding_idx=0)
        self.pos = PositionalEncoding(dm, 500, do)
        self.transformer = nn.Transformer(d_model=dm, nhead=nh, num_encoder_layers=nl,
                                          num_decoder_layers=nl, dim_feedforward=df, dropout=do, batch_first=True)
        self.fc = nn.Linear(dm, vs); self.drop = nn.Dropout(do)
    def make_mask(self, sz): return torch.triu(torch.ones(sz, sz, device=device), diagonal=1).bool()
    def forward(self, src, tgt):
        tm = self.make_mask(tgt.shape[1]); sp = (src==0); tp = (tgt==0)
        s = self.pos(self.drop(self.src_emb(src) * np.sqrt(self.d_model)))
        t = self.pos(self.drop(self.tgt_emb(tgt) * np.sqrt(self.d_model)))
        return self.fc(self.transformer(s, t, tgt_mask=tm, src_key_padding_mask=sp,
                                        tgt_key_padding_mask=tp, memory_key_padding_mask=sp))

# --- Greedy decoders ---
def greedy_decode_lstm(model, src, tokenizer, max_len):
    model.eval()
    with torch.no_grad():
        src = src.unsqueeze(0).to(device)
        enc_out, h, c = model.encoder(src)
        inp = torch.tensor([[tokenizer.token2idx[tokenizer.SOS_TOKEN]]]).to(device)
        tokens = []
        for _ in range(max_len):
            pred, h, c = model.decoder(inp, h, c, enc_out)
            next_t = pred.argmax(dim=1).item()
            if next_t == tokenizer.token2idx[tokenizer.EOS_TOKEN]: break
            tokens.append(next_t)
            inp = torch.tensor([[next_t]]).to(device)
    return tokens

def greedy_decode_transformer(model, src, tokenizer, max_len):
    model.eval()
    with torch.no_grad():
        src = src.unsqueeze(0).to(device)
        sos = tokenizer.token2idx[tokenizer.SOS_TOKEN]
        eos = tokenizer.token2idx[tokenizer.EOS_TOKEN]
        tgt_ids = [sos]
        for _ in range(max_len):
            tgt_t = torch.tensor([tgt_ids]).to(device)
            out = model(src, tgt_t)
            next_t = out[0, -1].argmax().item()
            if next_t == eos: break
            tgt_ids.append(next_t)
    return tgt_ids[1:]

print("\n✅ All classes and functions defined.")


✅ All classes and functions defined.


In [5]:
# Fix: rename state dict keys to match Notebook 2's class
import torch

def fix_transformer_keys(state_dict):
    """Rename Notebook 1 keys to match Notebook 2 class names."""
    mapping = {
        'src_embedding.weight': 'src_emb.weight',
        'tgt_embedding.weight': 'tgt_emb.weight',
        'positional_encoding.pe': 'pos.pe',
        'positional_encoding.dropout.weight': 'pos.dropout.weight',
        'fc_out.weight': 'fc.weight',
        'fc_out.bias': 'fc.bias',
        'dropout.weight': 'drop.weight',
    }
    new_dict = {}
    for k, v in state_dict.items():
        new_key = mapping.get(k, k)
        new_dict[new_key] = v
    return new_dict

# Load with fixed keys
lstm_model.load_state_dict(torch.load('best_lstm.pth', map_location=device))
transformer_model.load_state_dict(fix_transformer_keys(torch.load('best_transformer.pth', map_location=device)))
print("✅ Models loaded!")

✅ Models loaded!


In [7]:
lstm_model.load_state_dict(torch.load('best_lstm.pth', map_location=device))

# Fix transformer key names
sd = torch.load('best_transformer.pth', map_location=device)
new_sd = {}
for k, v in sd.items():
    k = k.replace('src_embedding', 'src_emb')
    k = k.replace('tgt_embedding', 'tgt_emb')
    k = k.replace('positional_encoding', 'pos')
    k = k.replace('fc_out', 'fc')
    new_sd[k] = v
transformer_model.load_state_dict(new_sd)
print("✅ Both models loaded!")

✅ Both models loaded!


In [8]:
# ============================================================
# Load data and models from Notebook 1
# ============================================================
import urllib.request

# Reload data
print("Loading data...")
if not os.path.exists('prosper_60k.txt'):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/hbprosper/symbolic-ai/main/data/seq2seq_data_60000.txt",
        "prosper_60k.txt"
    )

pairs = []
for line in open('prosper_60k.txt'):
    parts = line.strip().split('\t')
    if len(parts) == 2: pairs.append({'function': parts[0], 'taylor_expansion': parts[1]})

# Also load our SymPy data if available
if os.path.exists('our_sympy_data.csv'):
    our_data = pd.read_csv('our_sympy_data.csv')
    combined = pd.concat([our_data, pd.DataFrame(pairs)], ignore_index=True)
else:
    combined = pd.DataFrame(pairs)

df = combined.drop_duplicates(subset=['function']).reset_index(drop=True)

tokenizer = MathTokenizer()
tokenizer.build_vocab(df['function'].tolist() + df['taylor_expansion'].tolist())
VS = tokenizer.vocab_size

fl = [len(tokenizer.tokenize(f)) for f in df['function']]
tl = [len(tokenizer.tokenize(t)) for t in df['taylor_expansion']]
MAX_IN = int(np.percentile(fl, 95)) + 5
MAX_OUT = min(int(np.percentile(tl, 95)) + 5, 120)

df = df[(df['function'].apply(lambda f: len(tokenizer.tokenize(f)) <= MAX_IN-3)) &
        (df['taylor_expansion'].apply(lambda t: len(tokenizer.tokenize(t)) <= MAX_OUT-3))].reset_index(drop=True)

train_df, temp = train_test_split(df, test_size=0.15, random_state=SEED)
val_df, test_df = train_test_split(temp, test_size=0.33, random_state=SEED)
print(f"Data: {len(df)} pairs | Test: {len(test_df)}")

# Load models
EMBED=128; HIDDEN=256; NL=2; DROPOUT=0.3
lstm_model = Seq2SeqLSTM(VS, EMBED, HIDDEN, NL, DROPOUT).to(device)
transformer_model = Seq2SeqTransformer(VS, 256, 8, 4, 1024, DROPOUT).to(device)

lstm_model.load_state_dict(torch.load('best_lstm.pth', map_location=device))
transformer_model.load_state_dict(torch.load('best_transformer.pth', map_location=device))
print("\n✅ Models loaded!")

Loading data...
Data: 45023 pairs | Test: 2229


RuntimeError: Error(s) in loading state_dict for Seq2SeqTransformer:
	Missing key(s) in state_dict: "src_emb.weight", "tgt_emb.weight", "pos.pe", "fc.weight", "fc.bias". 
	Unexpected key(s) in state_dict: "src_embedding.weight", "tgt_embedding.weight", "positional_encoding.pe", "fc_out.weight", "fc_out.bias". 

---
## 1. Beam Search Implementation

Beam search keeps the top-k most promising partial sequences at each step. At the end, it picks the best complete sequence using **length-normalized log-probability** (to avoid favoring short sequences).

In [9]:
def beam_search_lstm(model, src, tokenizer, max_len, beam_width=5):
    """Beam search decoding for LSTM seq2seq."""
    model.eval()
    with torch.no_grad():
        src = src.unsqueeze(0).to(device)
        enc_out, h, c = model.encoder(src)
        sos = tokenizer.token2idx[tokenizer.SOS_TOKEN]
        eos = tokenizer.token2idx[tokenizer.EOS_TOKEN]

        # Each beam: (cumulative_log_prob, token_sequence, hidden_state, cell_state)
        beams = [(0.0, [sos], h, c)]
        completed = []

        for _ in range(max_len):
            candidates = []
            for score, seq, h_b, c_b in beams:
                if seq[-1] == eos:
                    completed.append((score / max(len(seq)-1, 1), seq[1:]))
                    continue
                inp = torch.tensor([[seq[-1]]]).to(device)
                pred, h_new, c_new = model.decoder(inp, h_b, c_b, enc_out)
                log_probs = torch.log_softmax(pred, dim=-1)
                topk_probs, topk_ids = log_probs.topk(beam_width)
                for i in range(beam_width):
                    candidates.append((
                        score + topk_probs[0, i].item(),
                        seq + [topk_ids[0, i].item()],
                        h_new, c_new
                    ))
            if not candidates: break
            candidates.sort(key=lambda x: x[0], reverse=True)
            beams = candidates[:beam_width]

        # Collect remaining beams
        for score, seq, _, _ in beams:
            completed.append((score / max(len(seq)-1, 1), seq[1:]))
        if not completed: return []
        completed.sort(key=lambda x: x[0], reverse=True)
        result = completed[0][1]
        if result and result[-1] == eos: result = result[:-1]
        return result


def beam_search_transformer(model, src, tokenizer, max_len, beam_width=5):
    """Beam search decoding for Transformer seq2seq."""
    model.eval()
    with torch.no_grad():
        src = src.unsqueeze(0).to(device)
        sos = tokenizer.token2idx[tokenizer.SOS_TOKEN]
        eos = tokenizer.token2idx[tokenizer.EOS_TOKEN]

        beams = [(0.0, [sos])]
        completed = []

        for _ in range(max_len):
            candidates = []
            for score, seq in beams:
                if seq[-1] == eos:
                    completed.append((score / max(len(seq)-1, 1), seq[1:]))
                    continue
                tgt = torch.tensor([seq]).to(device)
                out = model(src, tgt)
                log_probs = torch.log_softmax(out[0, -1], dim=-1)
                topk_probs, topk_ids = log_probs.topk(beam_width)
                for i in range(beam_width):
                    candidates.append((score + topk_probs[i].item(), seq + [topk_ids[i].item()]))
            if not candidates: break
            candidates.sort(key=lambda x: x[0], reverse=True)
            beams = candidates[:beam_width]

        for score, seq in beams:
            completed.append((score / max(len(seq)-1, 1), seq[1:]))
        if not completed: return []
        completed.sort(key=lambda x: x[0], reverse=True)
        result = completed[0][1]
        if result and result[-1] == eos: result = result[:-1]
        return result

print("✅ Beam search decoders ready!")

✅ Beam search decoders ready!


---
## 2. Evaluation: Greedy vs Beam Search

We evaluate both models with both decoding strategies on the full test set.

In [ ]:
def evaluate_full(model, greedy_fn, beam_fn, name, test_df, tok, max_in, max_out, beam_width=5):
    """Evaluate with both greedy and beam search, show comparison."""
    print(f"\n{'='*60}")
    print(f"Evaluating: {name}")
    print(f"{'='*60}")

    greedy_r = {'exact': 0, 'tok_c': 0, 'tok_t': 0, 'bleus': []}
    beam_r = {'exact': 0, 'tok_c': 0, 'tok_t': 0, 'bleus': []}
    details = []

    for idx in range(len(test_df)):
        row = test_df.iloc[idx]
        src = tok.pad_sequence(tok.encode(row['function']), max_in)
        src_t = torch.tensor(src, dtype=torch.long)
        true_str = row['taylor_expansion']
        true_idx = tok.encode(true_str)

        # Greedy
        g_idx = greedy_fn(model, src_t, tok, max_out)
        g_str = tok.decode(g_idx)
        # Beam
        b_idx = beam_fn(model, src_t, tok, max_out, beam_width)
        b_str = tok.decode(b_idx)

        g_exact = (g_str.replace(' ','') == true_str.replace(' ',''))
        b_exact = (b_str.replace(' ','') == true_str.replace(' ',''))
        if g_exact: greedy_r['exact'] += 1
        if b_exact: beam_r['exact'] += 1

        # Token accuracy
        for r_dict, p_idx in [(greedy_r, g_idx), (beam_r, b_idx)]:
            mn = min(len(p_idx), len(true_idx))
            r_dict['tok_c'] += sum(1 for i in range(mn) if p_idx[i] == true_idx[i])
            r_dict['tok_t'] += max(len(p_idx), len(true_idx))

        # BLEU-1
        t_tok = tok.tokenize(true_str)
        for r_dict, p_str in [(greedy_r, g_str), (beam_r, b_str)]:
            p_tok = tok.tokenize(p_str) if p_str else []
            if p_tok:
                pc = Counter(p_tok); tc = Counter(t_tok)
                r_dict['bleus'].append(sum(min(pc[t], tc[t]) for t in pc) / len(p_tok))
            else:
                r_dict['bleus'].append(0.0)

        details.append({'function': row['function'], 'true': true_str,
                       'greedy': g_str, 'beam': b_str,
                       'g_exact': g_exact, 'b_exact': b_exact})

        if (idx+1) % 500 == 0:
            print(f"  {idx+1}/{len(test_df)} done...")

    n = len(test_df)
    g = {'exact': greedy_r['exact']/n*100, 'tok': greedy_r['tok_c']/max(greedy_r['tok_t'],1)*100, 'bleu': np.mean(greedy_r['bleus'])*100}
    b = {'exact': beam_r['exact']/n*100, 'tok': beam_r['tok_c']/max(beam_r['tok_t'],1)*100, 'bleu': np.mean(beam_r['bleus'])*100}

    print(f"\n  {'Metric':<20s} {'Greedy':>10s} {'Beam-5':>10s} {'Improvement':>12s}")
    print(f"  {'-'*54}")
    print(f"  {'Exact Match %':<20s} {g['exact']:>9.2f}% {b['exact']:>9.2f}% {b['exact']-g['exact']:>+11.2f}%")
    print(f"  {'Token Accuracy %':<20s} {g['tok']:>9.2f}% {b['tok']:>9.2f}% {b['tok']-g['tok']:>+11.2f}%")
    print(f"  {'BLEU-1 %':<20s} {g['bleu']:>9.2f}% {b['bleu']:>9.2f}% {b['bleu']-g['bleu']:>+11.2f}%")

    # Show cases where beam search fixed greedy's mistakes
    ddf = pd.DataFrame(details)
    fixed = ddf[(ddf['b_exact']==True) & (ddf['g_exact']==False)]
    if len(fixed) > 0:
        print(f"\n  🔧 Beam search FIXED {len(fixed)} predictions that greedy got wrong:")
        for _, r in fixed.head(5).iterrows():
            print(f"    Function: {r['function'][:45]}")
            print(f"      Greedy: {r['greedy'][:55]}  ✗")
            print(f"      Beam:   {r['beam'][:55]}  ✓")

    return {'greedy': g, 'beam': b, 'details': ddf}


# Run evaluation
lstm_res = evaluate_full(lstm_model, greedy_decode_lstm, beam_search_lstm,
                         "LSTM + Attention", test_df, tokenizer, MAX_IN, MAX_OUT)

transformer_res = evaluate_full(transformer_model, greedy_decode_transformer, beam_search_transformer,
                                "Transformer", test_df, tokenizer, MAX_IN, MAX_OUT)


Evaluating: LSTM + Attention
  500/2229 done...
  1000/2229 done...
  1500/2229 done...
  2000/2229 done...

  Metric                   Greedy     Beam-5  Improvement
  ------------------------------------------------------
  Exact Match %            30.19%     30.64%       +0.45%
  Token Accuracy %         71.11%     71.95%       +0.84%
  BLEU-1 %                 94.65%     95.31%       +0.67%

  🔧 Beam search FIXED 23 predictions that greedy got wrong:
    Function: exp(6*x**3+4)+(9*x**2-5)*tan(4*x**2)
      Greedy: exp(4) + 20*x**2 + 6*x**3*exp(4) + 36*x**4  ✗
      Beam:   exp(4) - 20*x**2 + 6*x**3*exp(4) + 36*x**4  ✓
    Function: sin(-6*x+1)-ln(9*x**2+2)-exp(6*x**2-6)
      Greedy: sin(1) - exp( - 6) - log(2) - 6*x*cos(1) + x**2*( - 18*  ✗
      Beam:   sin(1) - exp( - 6) - log(2) - 6*x*cos(1) + x**2*( - 18*  ✓
    Function: (9*x**3+1)*ln(4*x**2)/(9*x**3-4)*sinh(7*x**3/
      Greedy: x**3*( - log(log))/log(2)/))  ✗
      Beam:   x**3*( - log(x)/2 - log(2)/2)  ✓
    Function: -(4

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


  500/2229 done...
  1000/2229 done...


---
## 3. Experiment: Teacher Forcing Decay for Transformer

### The Hypothesis

Teacher forcing decay made the LSTM robust — maybe it will work for the Transformer too?

### The Setup

- Epochs 1-10: Full teacher forcing (TF=1.0) for stable initial learning
- Epochs 11-20: Gradually reduce TF from 0.97 to ~0.75
- We run 20 epochs to show the effect clearly

### ⚠️ Spoiler: This Experiment Fails

We include this experiment specifically to show **what doesn't work** and **why**. This is an important finding — teacher forcing decay is not a universal fix for exposure bias.

In [ ]:
# ============================================================
# Transformer with Teacher Forcing Decay
# ============================================================

class Seq2SeqTransformerV2(nn.Module):
    """Transformer with scheduled sampling (teacher forcing decay) support."""
    def __init__(self, vs, dm, nh, nl, df, do):
        super().__init__()
        self.d_model = dm; self.vocab_size = vs
        self.src_emb = nn.Embedding(vs, dm, padding_idx=0)
        self.tgt_emb = nn.Embedding(vs, dm, padding_idx=0)
        self.pos = PositionalEncoding(dm, 500, do)
        self.transformer = nn.Transformer(
            d_model=dm, nhead=nh, num_encoder_layers=nl,
            num_decoder_layers=nl, dim_feedforward=df,
            dropout=do, batch_first=True
        )
        self.fc = nn.Linear(dm, vs)
        self.drop = nn.Dropout(do)

    def make_mask(self, sz):
        return torch.triu(torch.ones(sz, sz, device=device), diagonal=1).bool()

    def forward(self, src, tgt, teacher_forcing_ratio=1.0):
        if teacher_forcing_ratio >= 1.0:
            # Standard parallel forward pass (fast)
            tm = self.make_mask(tgt.shape[1])
            sp = (src == 0); tp = (tgt == 0)
            s = self.pos(self.drop(self.src_emb(src) * np.sqrt(self.d_model)))
            t = self.pos(self.drop(self.tgt_emb(tgt) * np.sqrt(self.d_model)))
            return self.fc(self.transformer(s, t, tgt_mask=tm,
                           src_key_padding_mask=sp, tgt_key_padding_mask=tp,
                           memory_key_padding_mask=sp))
        else:
            # Autoregressive with scheduled sampling (slow, for TF decay)
            sp = (src == 0)
            s = self.pos(self.drop(self.src_emb(src) * np.sqrt(self.d_model)))
            memory = self.transformer.encoder(s, src_key_padding_mask=sp)
            B, T = src.shape[0], tgt.shape[1]
            outputs = torch.zeros(B, T, self.vocab_size).to(device)
            generated = tgt[:, :1]  # Start with SOS
            for t_step in range(1, T):
                tm = self.make_mask(generated.shape[1])
                tp = (generated == 0)
                t_emb = self.pos(self.drop(self.tgt_emb(generated) * np.sqrt(self.d_model)))
                out = self.transformer.decoder(t_emb, memory, tgt_mask=tm,
                        tgt_key_padding_mask=tp, memory_key_padding_mask=sp)
                pred = self.fc(out[:, -1:, :])
                outputs[:, t_step] = pred.squeeze(1)
                if random.random() < teacher_forcing_ratio:
                    next_token = tgt[:, t_step:t_step+1]
                else:
                    next_token = pred.argmax(dim=-1)
                generated = torch.cat([generated, next_token], dim=1)
            return outputs

print("Seq2SeqTransformerV2 defined (with TF decay support).")

In [ ]:
print("="*60)
print("EXPERIMENT: TRANSFORMER WITH TEACHER FORCING DECAY")
print("="*60)
print("\nHypothesis: TF decay will fix the Transformer's exposure bias,")
print("just like it did for the LSTM.")
print("\nExpected: Val loss continues to decrease after TF drops below 1.0")
print("Actual:   See below...\n")

# Use small batch size for TF decay (autoregressive loop uses more memory)
class TaylorDataset(Dataset):
    def __init__(self, df, tok, mi, mo):
        self.data = df.reset_index(drop=True); self.tok = tok; self.mi = mi; self.mo = mo
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        r = self.data.iloc[i]
        return {'src': torch.tensor(self.tok.pad_sequence(self.tok.encode(r['function']), self.mi), dtype=torch.long),
                'tgt': torch.tensor(self.tok.pad_sequence(self.tok.encode(r['taylor_expansion'], add_sos=True, add_eos=True), self.mo), dtype=torch.long)}

BS_LARGE = 128  # for full TF epochs
BS_SMALL = 32   # for TF decay epochs (needs more memory)

train_loader_large = DataLoader(TaylorDataset(train_df, tokenizer, MAX_IN, MAX_OUT),
                                batch_size=BS_LARGE, shuffle=True, num_workers=4, pin_memory=True)
train_loader_small = DataLoader(TaylorDataset(train_df, tokenizer, MAX_IN, MAX_OUT),
                                batch_size=BS_SMALL, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(TaylorDataset(val_df, tokenizer, MAX_IN, MAX_OUT),
                        batch_size=BS_LARGE, num_workers=4, pin_memory=True)

# Initialize fresh Transformer v2
tf_model = Seq2SeqTransformerV2(VS, 256, 8, 4, 1024, DROPOUT).to(device)
opt_tf = torch.optim.Adam(tf_model.parameters(), lr=5e-4, betas=(0.9, 0.98), eps=1e-9)
crit_tf = nn.CrossEntropyLoss(ignore_index=0)

TF_EPOCHS = 20  # 10 full TF + 10 with decay (enough to show the collapse)
tf_history = {'train': [], 'val': [], 'tf_ratio': []}

for epoch in range(TF_EPOCHS):
    tf_model.train()
    tl = 0

    if epoch < 10:
        tf_ratio = 1.0
        loader = train_loader_large
    else:
        tf_ratio = max(0.3, 1.0 - (epoch - 10) * 0.03)
        loader = train_loader_small

    for batch in loader:
        src, tgt = batch['src'].to(device), batch['tgt'].to(device)
        opt_tf.zero_grad()
        out = tf_model(src, tgt[:, :-1], teacher_forcing_ratio=tf_ratio)
        out = out.contiguous().view(-1, VS)
        loss = crit_tf(out, tgt[:, 1:].contiguous().view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(tf_model.parameters(), 1.0)
        opt_tf.step()
        tl += loss.item()
        if tf_ratio < 1.0:
            torch.cuda.empty_cache()
    tl /= len(loader)

    tf_model.eval()
    vl = 0
    with torch.no_grad():
        for batch in val_loader:
            src, tgt = batch['src'].to(device), batch['tgt'].to(device)
            out = tf_model(src, tgt[:, :-1], teacher_forcing_ratio=1.0)
            out = out.contiguous().view(-1, VS)
            vl += crit_tf(out, tgt[:, 1:].contiguous().view(-1)).item()
    vl /= len(val_loader)

    tf_history['train'].append(tl)
    tf_history['val'].append(vl)
    tf_history['tf_ratio'].append(tf_ratio)

    marker = '← TF decay starts here!' if epoch == 10 else ''
    print(f"Ep {epoch+1:2d}/{TF_EPOCHS} | Train: {tl:.4f} | Val: {vl:.4f} | TF: {tf_ratio:.2f}  {marker}")

print("\n" + "="*60)
print("RESULT: The Transformer collapsed when TF dropped below 1.0")
print("="*60)

### Why Did TF Decay Fail for the Transformer?

Three compounding factors:

1. **Attention amplifies errors.** In the LSTM, a wrong token affects the next hidden state — a local disturbance. In the Transformer, a wrong token is attended to by every position across every head across every layer. One wrong token pollutes the entire representation.

2. **Abrupt transition.** The model spent 10 epochs optimized for perfect inputs. Even going from 100% to 97% correct tokens (TF=0.97) means ~4 wrong tokens per 120-token sequence. Each one disrupts all attention computations.

3. **The autoregressive loop is fundamentally different** from the standard parallel Transformer forward pass. It's much slower, uses more memory, and produces different gradient signals.

### The Lesson

**Teacher forcing decay is NOT a universal fix for exposure bias:**
- ✅ Works for **LSTMs** — the recurrent hidden state absorbs perturbations gradually
- ❌ Breaks **Transformers** — multi-head attention amplifies perturbations catastrophically

**The right approach for each architecture:**
- LSTM → fix during **training** (teacher forcing decay)
- Transformer → fix during **inference** (beam search)

---
## 4. Final Comparison & Visualization

In [ ]:
# ============================================================
# Final Comparison Table
# ============================================================
print("\n" + "="*80)
print("FINAL MODEL COMPARISON — ALL EXPERIMENTS")
print("="*80)
print(f"{'Model':<30s} {'Decode':<8s} {'Exact%':>8s} {'TokAcc%':>9s} {'BLEU%':>8s}")
print("-"*65)
print(f"{'Prosper Baseline (3890 ep)':<30s} {'greedy':<8s} {'N/A':>8s} {'N/A':>9s} {'N/A':>8s}")
print(f"{'LSTM + Attention':<30s} {'greedy':<8s} {lstm_res['greedy']['exact']:>7.2f}% {lstm_res['greedy']['tok']:>8.2f}% {lstm_res['greedy']['bleu']:>7.2f}%")
print(f"{'LSTM + Attention':<30s} {'beam-5':<8s} {lstm_res['beam']['exact']:>7.2f}% {lstm_res['beam']['tok']:>8.2f}% {lstm_res['beam']['bleu']:>7.2f}%")
print(f"{'Transformer':<30s} {'greedy':<8s} {transformer_res['greedy']['exact']:>7.2f}% {transformer_res['greedy']['tok']:>8.2f}% {transformer_res['greedy']['bleu']:>7.2f}%")
print(f"{'Transformer':<30s} {'beam-5':<8s} {transformer_res['beam']['exact']:>7.2f}% {transformer_res['beam']['tok']:>8.2f}% {transformer_res['beam']['bleu']:>7.2f}%")
print(f"{'Transformer + TF Decay':<30s} {'—':<8s} {'COLLAPSED':>8s} {'—':>9s} {'—':>8s}")
print("="*65)

In [ ]:
# ============================================================
# Visualization]
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('FASEROH: Complete Experimental Results', fontsize=16, fontweight='bold', y=1.02)

# 1. TF Decay Collapse
epochs_range = range(1, len(tf_history['val'])+1)
axes[0,0].plot(epochs_range, tf_history['val'], 'r-o', linewidth=2, markersize=4, label='Val Loss')
axes[0,0].plot(epochs_range, tf_history['train'], 'b-o', linewidth=2, markersize=4, label='Train Loss')
axes[0,0].axvline(x=10.5, color='gray', linestyle='--', alpha=0.7, label='TF decay starts')
axes[0,0].set_title('Transformer + TF Decay: The Collapse', fontweight='bold')
axes[0,0].set_xlabel('Epoch'); axes[0,0].set_ylabel('Loss')
axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)
axes[0,0].annotate('TF drops below 1.0\n→ instant collapse', xy=(11, tf_history['val'][10]),
                   xytext=(13, max(tf_history['val'])*0.7),
                   arrowprops=dict(arrowstyle='->', color='red'), fontsize=9, color='red')

# 2. Greedy vs Beam Search comparison
models = ['LSTM', 'Transformer']
greedy_exact = [lstm_res['greedy']['exact'], transformer_res['greedy']['exact']]
beam_exact = [lstm_res['beam']['exact'], transformer_res['beam']['exact']]
x_pos = np.arange(len(models))
axes[0,1].bar(x_pos - 0.2, greedy_exact, 0.35, label='Greedy', color='#F44336', edgecolor='black')
axes[0,1].bar(x_pos + 0.2, beam_exact, 0.35, label='Beam Search (k=5)', color='#4CAF50', edgecolor='black')
axes[0,1].set_xticks(x_pos); axes[0,1].set_xticklabels(models)
axes[0,1].set_title('Exact Match: Greedy vs Beam Search', fontweight='bold')
axes[0,1].set_ylabel('Exact Match %'); axes[0,1].legend()
for i, (g, b) in enumerate(zip(greedy_exact, beam_exact)):
    axes[0,1].annotate(f'{g:.1f}%', xy=(i-0.2, g), ha='center', va='bottom', fontsize=10)
    axes[0,1].annotate(f'{b:.1f}%', xy=(i+0.2, b), ha='center', va='bottom', fontsize=10)

# 3. All metrics comparison
metrics = ['Exact Match', 'Token Acc', 'BLEU-1']
lstm_greedy_vals = [lstm_res['greedy']['exact'], lstm_res['greedy']['tok'], lstm_res['greedy']['bleu']]
lstm_beam_vals = [lstm_res['beam']['exact'], lstm_res['beam']['tok'], lstm_res['beam']['bleu']]
trans_beam_vals = [transformer_res['beam']['exact'], transformer_res['beam']['tok'], transformer_res['beam']['bleu']]

x_pos = np.arange(len(metrics))
w = 0.25
axes[1,0].bar(x_pos - w, lstm_greedy_vals, w, label='LSTM Greedy', color='#2196F3', edgecolor='black')
axes[1,0].bar(x_pos, lstm_beam_vals, w, label='LSTM Beam', color='#1565C0', edgecolor='black')
axes[1,0].bar(x_pos + w, trans_beam_vals, w, label='Transformer Beam', color='#FF9800', edgecolor='black')
axes[1,0].set_xticks(x_pos); axes[1,0].set_xticklabels(metrics)
axes[1,0].set_title('All Models & Methods Compared', fontweight='bold')
axes[1,0].set_ylabel('Percentage'); axes[1,0].legend(fontsize=9)

# 4. TF ratio vs val loss
ax2 = axes[1,1]
ax2.plot(tf_history['tf_ratio'], tf_history['val'], 'ro-', markersize=6, linewidth=2)
ax2.set_xlabel('Teacher Forcing Ratio')
ax2.set_ylabel('Validation Loss')
ax2.set_title('TF Ratio vs Val Loss (Transformer)', fontweight='bold')
ax2.invert_xaxis()  # higher TF on left
ax2.grid(True, alpha=0.3)
ax2.annotate('Stable at TF=1.0', xy=(1.0, tf_history['val'][0]), fontsize=9, color='green')
ax2.annotate('Collapse at TF<1.0', xy=(tf_history['tf_ratio'][10], tf_history['val'][10]),
            fontsize=9, color='red')

plt.tight_layout()
plt.savefig('beam_search_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nPlot saved: beam_search_results.png")

In [ ]:
# Save all results
pickle.dump({
    'lstm_res': lstm_res,
    'transformer_res': transformer_res,
    'tf_history': tf_history,
}, open('notebook2_results.pkl', 'wb'))

print("="*60)
print("✅ Notebook 2 complete!")
print("="*60)
print("\nKey findings:")
print(f"  1. LSTM + Beam Search is our best model")
print(f"  2. Beam search improves inference without retraining")
print(f"  3. TF decay breaks Transformers — attention amplifies errors")
print(f"  4. Different architectures need different solutions to exposure bias")
print(f"\nSaved: beam_search_results.png, notebook2_results.pkl")